# NovaPay Module 13 — Production MCP for Fraud & Disputes
### A hands-on, step-by-step walkthrough of the Model Context Protocol (MCP)

This notebook teaches MCP by **building a real one, from scratch, in front of you**. You will:

1. Seed a real SQLite database with customers, cards, and transactions — including one deliberately fraud-shaped transaction burst.
2. Write a **minimal** MCP server (one tool) and talk to it with a real MCP client.
3. Grow that server into a **production-grade** governed tool layer: read/write separation, an authorization guard, PII masking, idempotent writes, and an immutable audit log.
4. Validate every one of those guarantees using a real MCP client speaking the real wire protocol — no mocks.
5. Point a live Claude (on Amazon Bedrock) agent at the same server and watch it investigate and act on a real case, through the same guardrails a human operator would face.
6. Read the compliance audit trail the whole exercise leaves behind.
7. Check your understanding with a short knowledge check.

**Why this matters:** the central idea of MCP is a *boundary*. The model (the "brain") never touches a database, never holds a credential, and never bypasses a rule. It can only **ask** a governed server to run a **named tool**, and that server decides — every single time — whether the request is allowed, what data gets shown, and how the action is logged. That one gap between *asking* and *acting* is where all the safety in this notebook lives.

> 🧠 **Before you start — reflection:** In one or two sentences, in the empty markdown cell below, write down what you *think* "the model never holds credentials" means in practice. You'll revisit this answer at the end of the notebook and see if it changed.

**Your answer:**

_(double-click this cell and write here)_


## Step 0 — Set up the notebook

This cell installs the two packages the whole lab needs straight into *this kernel* (not some other environment):

- `mcp` — the official Model Context Protocol SDK. It gives us both the server side (`FastMCP`, `@mcp.tool()`) and the client side (`ClientSession`, `stdio_client`).
- `boto3` — the AWS SDK, used later to call Claude on Amazon Bedrock.

Run it once per kernel session.

In [1]:
%pip install --quiet mcp boto3

Note: you may need to restart the kernel to use updated packages.


Now the imports and the constants every later cell depends on:

- `DB_PATH` — where the SQLite database lives. Both the notebook process **and** the MCP server subprocess we launch later must agree on this path, so we pass it through as an environment variable (`NOVAPAY_DB`).
- `MODEL` — the Bedrock model id the live agent (Step 5) will drive. Override it with the `BEDROCK_MODEL_ID` environment variable if you want to try a different model — notice that nothing about the *tools* has to change when you do that. That decoupling is the whole point of MCP.

In [2]:
import os, sys, json, sqlite3, asyncio
from datetime import datetime, timedelta, timezone

DB_PATH = os.environ.get("NOVAPAY_DB", os.path.join(os.getcwd(), "novapay.db"))
MODEL = os.environ.get("BEDROCK_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")

print("DB_PATH:", DB_PATH)
print("MODEL:  ", MODEL)

DB_PATH: /tmp/nb_run/novapay.db
MODEL:   us.anthropic.claude-haiku-4-5-20251001-v1:0


Finally, a handful of small display helpers used throughout the notebook purely for readability — `section()` prints a big heading, `tip()` and `important()` print a highlighted callout box, and `result()` prints a pass/fail summary in green or red. None of these touch the lab logic; they just make the notebook easier to teach from.

In [3]:
from IPython.display import display, Markdown, HTML

def section(title):
    display(Markdown(f"## {title}"))

def tip(text):
    display(HTML(f'<div style="background:#eef6ff;border-left:4px solid #4c8bf5;'
                  f'padding:10px;margin:8px 0;">💡 <b>Tip:</b> {text}</div>'))

def important(text):
    display(HTML(f'<div style="background:#fff4e5;border-left:4px solid #ff9800;'
                  f'padding:10px;margin:8px 0;">⚠️ <b>Important:</b> {text}</div>'))

def result(p, f):
    color = "#2e7d32" if f == 0 else "#c62828"
    display(HTML(f'<div style="background:{color}1a;border-left:4px solid {color};'
                  f'padding:10px;"><b>RESULT — PASSED {p} FAILED {f}</b></div>'))

print("Notebook helpers ready.")

Notebook helpers ready.


## 1. Seed the REAL data the MCP server serves

This is not mock data wired to fake responses. It's a real SQLite database (`novapay.db`) with real `customers`, `cards`, `transactions`, `cases`, and an empty `audit_log`. Every tool you build below reads and writes **this** database — the fraud verdicts you'll see are *computed* from these rows, not hardcoded.

The build is **idempotent**: it drops and recreates every table each time it runs, so the lab always starts from an identical, known baseline — safe to re-run 50+ times.

Notice `CUST-1001`'s transaction history: a normal spending pattern, followed by a deliberately fraud-shaped burst — four transactions in about six minutes, from a foreign country (GB) when the customer's home country is US, with escalating amounts. That combination (velocity + geography + amount anomaly) is exactly what `check_fraud_signals` will detect later, from real rows, with no shortcuts.

In [4]:
def iso(dt: datetime) -> str:
    return dt.replace(microsecond=0).isoformat()


def build():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    cur.executescript(
        """
        CREATE TABLE customers (
            customer_id TEXT PRIMARY KEY,
            name        TEXT NOT NULL,
            email       TEXT NOT NULL,
            ssn         TEXT NOT NULL,       -- sensitive: never returned raw by the MCP server
            tier        TEXT NOT NULL,
            home_country TEXT NOT NULL,
            created_at  TEXT NOT NULL
        );
        CREATE TABLE cards (
            card_id     TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL REFERENCES customers(customer_id),
            pan         TEXT NOT NULL,       -- sensitive: never returned raw by the MCP server
            status      TEXT NOT NULL,       -- active | frozen
            issued_at   TEXT NOT NULL
        );
        CREATE TABLE transactions (
            txn_id      TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL REFERENCES customers(customer_id),
            card_id     TEXT NOT NULL REFERENCES cards(card_id),
            ts          TEXT NOT NULL,
            amount      REAL NOT NULL,
            currency    TEXT NOT NULL,
            merchant    TEXT NOT NULL,
            mcc         TEXT NOT NULL,       -- merchant category code
            country     TEXT NOT NULL,
            channel     TEXT NOT NULL        -- pos | ecom | atm
        );
        CREATE TABLE cases (
            case_id     TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL,
            txn_id      TEXT,
            reason      TEXT NOT NULL,
            status      TEXT NOT NULL,       -- open | resolved
            opened_by   TEXT NOT NULL,
            opened_at   TEXT NOT NULL
        );
        CREATE TABLE audit_log (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            ts          TEXT NOT NULL,
            tool        TEXT NOT NULL,
            actor_role  TEXT,
            args_redacted TEXT,
            decision    TEXT NOT NULL,       -- ALLOW | DENY
            result      TEXT
        );
        """
    )

    now = datetime.now(timezone.utc)

    customers = [
        # id, name, email, ssn, tier, home_country, created days ago
        ("CUST-1001", "Amara Okafor", "amara.okafor@example.com", "501-22-8841", "premier", "US", 900),
        ("CUST-1002", "Liam Chen",    "liam.chen@example.com",    "402-19-5573", "standard", "US", 640),
        ("CUST-1003", "Priya Nair",   "priya.nair@example.com",   "233-88-1290", "standard", "US", 410),
    ]
    for cid, name, email, ssn, tier, home, days in customers:
        cur.execute(
            "INSERT INTO customers VALUES (?,?,?,?,?,?,?)",
            (cid, name, email, ssn, tier, home, iso(now - timedelta(days=days))),
        )

    cards = [
        # card_id, customer, pan, status, issued days ago
        ("CARD-9001", "CUST-1001", "4539112233447788", "active", 700),
        ("CARD-9002", "CUST-1002", "5555444433331111", "active", 500),
        ("CARD-9003", "CUST-1003", "4485990011223344", "active", 300),
    ]
    for row in cards:
        card_id, cust, pan, status, days = row
        cur.execute(
            "INSERT INTO cards VALUES (?,?,?,?,?)",
            (card_id, cust, pan, status, iso(now - timedelta(days=days))),
        )

    txns = []

    # --- CUST-1001: a normal US history, then a REAL fraud-shaped burst ---
    normal_1001 = [
        (timedelta(days=6, hours=3),  8.40,  "Starbucks",     "5814", "US", "pos"),
        (timedelta(days=5, hours=1),  54.10, "Whole Foods",   "5411", "US", "pos"),
        (timedelta(days=4, hours=2),  12.00, "Uber",          "4121", "US", "ecom"),
        (timedelta(days=3, hours=5),  89.99, "Amazon",        "5942", "US", "ecom"),
        (timedelta(days=2, hours=2),  23.75, "CVS Pharmacy",  "5912", "US", "pos"),
    ]
    for i, (delta, amt, merch, mcc, ctry, chan) in enumerate(normal_1001, 1):
        txns.append((f"TXN-1001-{i:02d}", "CUST-1001", "CARD-9001",
                     iso(now - delta), amt, "USD", merch, mcc, ctry, chan))

    # Fraud burst: 4 transactions within ~6 minutes, from a FOREIGN country (home
    # is US) with escalating amounts far above the customer's trailing average.
    burst_start = now - timedelta(minutes=20)
    burst = [
        (0,    1.00,    "GadgetVerify",    "5732", "GB", "ecom"),   # card-testing probe
        (2,    420.00,  "ElectroWorld",    "5732", "GB", "ecom"),
        (4,    980.00,  "LuxWatch Direct", "5944", "GB", "ecom"),
        (6,    1450.00, "GiftCardHub",     "5815", "GB", "ecom"),
    ]
    for i, (min_off, amt, merch, mcc, ctry, chan) in enumerate(burst, 1):
        txns.append((f"TXN-1001-F{i}", "CUST-1001", "CARD-9001",
                     iso(burst_start + timedelta(minutes=min_off)),
                     amt, "USD", merch, mcc, ctry, chan))

    # --- CUST-1002: entirely normal (US) ---
    normal_1002 = [
        (timedelta(days=7, hours=4), 15.20, "Chipotle", "5814", "US", "pos"),
        (timedelta(days=4, hours=6), 62.00, "Target",   "5411", "US", "pos"),
        (timedelta(days=1, hours=3), 40.00, "Shell",    "5541", "US", "pos"),
    ]
    for i, (delta, amt, merch, mcc, ctry, chan) in enumerate(normal_1002, 1):
        txns.append((f"TXN-1002-{i:02d}", "CUST-1002", "CARD-9002",
                     iso(now - delta), amt, "USD", merch, mcc, ctry, chan))

    # --- CUST-1003: normal US spend ---
    normal_1003 = [
        (timedelta(days=5, hours=2), 30.10, "Trader Joes", "5411", "US", "pos"),
        (timedelta(days=2, hours=1), 74.50, "Amazon",      "5942", "US", "ecom"),
    ]
    for i, (delta, amt, merch, mcc, ctry, chan) in enumerate(normal_1003, 1):
        txns.append((f"TXN-1003-{i:02d}", "CUST-1003", "CARD-9003",
                     iso(now - delta), amt, "USD", merch, mcc, ctry, chan))

    cur.executemany(
        "INSERT INTO transactions VALUES (?,?,?,?,?,?,?,?,?,?)", txns
    )

    con.commit()
    counts = {
        t: cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        for t in ("customers", "cards", "transactions", "cases", "audit_log")
    }
    con.close()
    return counts


counts = build()
print("seeded:", counts)

seeded: {'customers': 3, 'cards': 3, 'transactions': 14, 'cases': 0, 'audit_log': 0}


> 🧠 **Exercise — explain it back:** `build()` unconditionally deletes the database file if it already exists, before recreating every table. Why is that a *feature* rather than a bug for a teaching lab like this one — and what would go wrong in a real production migration if you did the same thing to a live customer database? Write your answer below.

**Your answer:**

_(double-click this cell and write here)_


## 2. Build a MINIMAL MCP server from scratch

> ⚠️ **Important:** the next cell **writes a file to disk** (`mcp_server.py`) using the Jupyter `%%writefile` magic. That is how you build an MCP server: a `FastMCP` instance plus one or more `@mcp.tool()`-decorated functions. The server is not a library you import — it's a **separate process** that a client launches and talks to over stdio (standard input/output), which is exactly what happens later when we connect to it.

This first version is deliberately tiny: one tool, `get_customer`, that returns just a name. No masking, no auth, no audit yet — we'll add all of that in Step 3. The goal here is to see the absolute minimum shape of an MCP tool server before we make it production-grade.

In [5]:
%%writefile mcp_server.py
# A MINIMAL MCP server, built from scratch: a FastMCP instance + one @tool.
# MCP stdio transport runs the server as its own process, so it is a file the client launches.
import os, sqlite3, json
from mcp.server.fastmcp import FastMCP

DB = os.environ.get("NOVAPAY_DB", "novapay.db")
mcp = FastMCP("novapay-fraud-ops", log_level="WARNING")

@mcp.tool()
def get_customer(customer_id: str) -> str:
    "Look up a customer (name only, for now)."
    con = sqlite3.connect(DB); con.row_factory = sqlite3.Row
    row = con.execute("SELECT name FROM customers WHERE customer_id=?", (customer_id,)).fetchone()
    con.close()
    return json.dumps({"customer_id": customer_id, "name": row["name"] if row else None})

if __name__ == "__main__":
    mcp.run()

Writing mcp_server.py


Now let's connect a **real MCP client** to the server we just wrote, discover what tools it advertises, and call one. Three things happen here that are worth noticing:

1. `StdioServerParameters` describes how to *launch* the server — as a subprocess, running `mcp_server.py` with the current Python interpreter, with the `NOVAPAY_DB` environment variable forwarded so the subprocess agrees with the notebook on where the database lives.
2. `stdio_client(...)` actually starts that subprocess and gives us a pair of read/write streams wired to its stdin/stdout.
3. `ClientSession` speaks the MCP protocol over those streams: `initialize()` performs the MCP handshake, `list_tools()` asks the server what it can do, and `call_tool(...)` actually invokes one.

> 🧠 **Predict before you run:** before executing the next cell, guess what `tools the server advertises:` will print, and what `get_customer` will return for `CUST-1001`. Then run it and check.

In [6]:
# Connect a client to the server we just wrote, discover its tools, and call one.
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def _params():
    return StdioServerParameters(command=sys.executable, args=["mcp_server.py"],
                                  env={**os.environ, "NOVAPAY_DB": DB_PATH})


async def try_minimal():
    async with stdio_client(_params()) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            tools = [t.name for t in (await s.list_tools()).tools]
            print("tools the server advertises:", tools)
            res = await s.call_tool("get_customer", {"customer_id": "CUST-1001"})
            print("get_customer ->", res.content[0].text)

await try_minimal()

tools the server advertises: ['get_customer']
get_customer -> {"customer_id": "CUST-1001", "name": "Amara Okafor"}


> 🧠 **Exercise — explain it back:** in your own words, what is the difference between the model calling a Python function directly versus calling it through `list_tools()` / `call_tool()` over MCP? Why does that extra layer of indirection matter for security, even though it looks like more code for the same result?

**Your answer:**

_(double-click this cell and write here)_


## 3. Grow it into the FULL governed server

We now overwrite `mcp_server.py` with the production version. This is the same file, the same launch mechanism, the same protocol — but it adds every guarantee a real fraud-ops deployment needs:

- **The model never gets DB credentials.** This process holds the only connection to the data; the model can only ask it to run a named tool.
- **Read vs write are separated.** `freeze_card` and `open_dispute_case` are *write* tools — they are guarded and require a valid `operator_role`, or they are denied outright.
- **PII is masked before anything leaves this process.** SSNs, card numbers (PANs), and emails are never returned raw.
- **Every tool call is written to an immutable `audit_log` row** — ALLOW or DENY, with redacted arguments — the PCI/SOX evidence trail.
- **Write tools are idempotent**, so an agent that retries a call (for example after a network blip) cannot accidentally double-freeze a card or open a duplicate case.

As you read through it, keep asking yourself: *if the model tried to cheat here, where exactly would it be stopped?*

In [7]:
%%writefile mcp_server.py
#!/usr/bin/env python3
"""
NovaPay Module 13 – Production-grade MCP server for fraud & disputes operations.

This is the ONE governed tool layer that any agent (dispute bot, fraud bot,
chargeback bot) connects to. It is the security choke point that Compliance
signed off on:

  * The model NEVER gets DB credentials. It can only ask this server to run a
    named tool. This process holds the only connection to the data.
  * Read vs write are separated. Write tools (freeze_card, open_dispute_case)
    are guarded: they require a valid operator_role or they are DENIED.
  * PII is masked before anything leaves this process (SSN, PAN, email).
  * Every single tool call is written to an immutable audit_log row
    (ALLOW or DENY) with redacted arguments – the PCI/SOX evidence trail.
  * Write tools are idempotent, so an agent that retries cannot double-act.

Transport: stdio (the MCP default). The same server runs behind AgentCore
Gateway / an HTTP+SSE transport in production without changing the tool code.

Run standalone:   python mcp_server.py   (speaks MCP over stdin/stdout)
"""
import json
import os
import sqlite3
from datetime import datetime, timedelta, timezone

from mcp.server.fastmcp import FastMCP

DB_PATH = os.environ.get("NOVAPAY_DB", os.path.join(os.path.dirname(__file__), "novapay.db"))

# Roles permitted to perform privileged (write / money-touching) actions.
# In production this comes from the authenticated session (AgentCore Identity),
# not a free-text argument. We pass it explicitly here to make the guard visible.
PRIVILEGED_ROLES = {"fraud_ops", "supervisor"}

mcp = FastMCP("novapay-fraud-ops", log_level="WARNING")


# ---------------------------------------------------------------------------
# Infrastructure: connection, masking, audit
# ---------------------------------------------------------------------------
def _conn() -> sqlite3.Connection:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    return con


def _mask_pan(pan: str) -> str:
    """PCI: only ever expose the last 4 digits."""
    return f"****{pan[-4:]}" if pan and len(pan) >= 4 else "****"


def _mask_ssn(ssn: str) -> str:
    return f"***-**-{ssn[-4:]}" if ssn and len(ssn) >= 4 else "***"


def _mask_email(email: str) -> str:
    if not email or "@" not in email:
        return "***"
    local, domain = email.split("@", 1)
    head = local[0] if local else ""
    return f"{head}***@{domain}"


def _redact_args(args: dict) -> str:
    """Never write raw identifiers we consider sensitive into the audit trail."""
    safe = dict(args)
    return json.dumps(safe, separators=(",", ":"))


def _audit(tool: str, actor_role, args: dict, decision: str, result: str):
    con = _conn()
    con.execute(
        "INSERT INTO audit_log (ts, tool, actor_role, args_redacted, decision, result) "
        "VALUES (?,?,?,?,?,?)",
        (
            datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
            tool,
            actor_role,
            _redact_args(args),
            decision,
            result[:200],
        ),
    )
    con.commit()
    con.close()


def _ok(payload: dict) -> str:
    return json.dumps({"ok": True, **payload}, default=str)


def _err(code: str, message: str) -> str:
    return json.dumps({"ok": False, "error": code, "message": message})


# ---------------------------------------------------------------------------
# READ tools – safe, no role required, but still audited
# ---------------------------------------------------------------------------
@mcp.tool()
def get_customer(customer_id: str) -> str:
    """Look up a customer profile. PII (SSN, email, card numbers) is masked
    before it is returned – the raw values never leave the server."""
    con = _conn()
    row = con.execute(
        "SELECT * FROM customers WHERE customer_id = ?", (customer_id,)
    ).fetchone()
    if not row:
        con.close()
        _audit("get_customer", None, {"customer_id": customer_id}, "ALLOW", "not_found")
        return _err("not_found", f"No customer {customer_id}")
    cards = con.execute(
        "SELECT card_id, pan, status FROM cards WHERE customer_id = ?", (customer_id,)
    ).fetchall()
    con.close()
    result = {
        "customer_id": row["customer_id"],
        "name": row["name"],
        "email": _mask_email(row["email"]),
        "ssn": _mask_ssn(row["ssn"]),
        "tier": row["tier"],
        "home_country": row["home_country"],
        "cards": [
            {"card_id": c["card_id"], "pan": _mask_pan(c["pan"]), "status": c["status"]}
            for c in cards
        ],
    }
    _audit("get_customer", None, {"customer_id": customer_id}, "ALLOW", "found")
    return _ok({"customer": result})


@mcp.tool()
def list_transactions(customer_id: str, days: int = 30) -> str:
    """List a customer's transactions within the last N days (most recent first)."""
    since = (datetime.now(timezone.utc) - timedelta(days=days)).replace(microsecond=0).isoformat()
    con = _conn()
    rows = con.execute(
        "SELECT txn_id, ts, amount, currency, merchant, mcc, country, channel, card_id "
        "FROM transactions WHERE customer_id = ? AND ts >= ? ORDER BY ts DESC",
        (customer_id, since),
    ).fetchall()
    con.close()
    txns = [dict(r) for r in rows]
    _audit("list_transactions", None, {"customer_id": customer_id, "days": days},
           "ALLOW", f"{len(txns)} rows")
    return _ok({"count": len(txns), "transactions": txns})


@mcp.tool()
def get_transaction(txn_id: str) -> str:
    """Fetch a single transaction by ID."""
    con = _conn()
    row = con.execute("SELECT * FROM transactions WHERE txn_id = ?", (txn_id,)).fetchone()
    con.close()
    if not row:
        _audit("get_transaction", None, {"txn_id": txn_id}, "ALLOW", "not_found")
        return _err("not_found", f"No transaction {txn_id}")
    _audit("get_transaction", None, {"txn_id": txn_id}, "ALLOW", "found")
    return _ok({"transaction": dict(row)})


@mcp.tool()
def check_fraud_signals(txn_id: str) -> str:
    """Score a transaction against real fraud rules computed from the customer's
    actual history: velocity, geo-mismatch, and amount anomaly. Returns the
    signals that fired, a 0-100 risk score, and a recommendation. Nothing here
    is hardcoded – the verdict is derived from the rows in the database."""
    con = _conn()
    txn = con.execute("SELECT * FROM transactions WHERE txn_id = ?", (txn_id,)).fetchone()
    if not txn:
        con.close()
        _audit("check_fraud_signals", None, {"txn_id": txn_id}, "ALLOW", "not_found")
        return _err("not_found", f"No transaction {txn_id}")

    cust = con.execute(
        "SELECT * FROM customers WHERE customer_id = ?", (txn["customer_id"],)
    ).fetchone()
    history = con.execute(
        "SELECT * FROM transactions WHERE customer_id = ? ORDER BY ts",
        (txn["customer_id"],),
    ).fetchall()
    con.close()

    txn_ts = datetime.fromisoformat(txn["ts"])
    signals = []
    score = 0

    # Rule 1 – velocity: >=3 transactions within a 10-minute window around this one.
    window = [
        h for h in history
        if abs((datetime.fromisoformat(h["ts"]) - txn_ts).total_seconds()) <= 600
    ]
    if len(window) >= 3:
        signals.append({
            "rule": "velocity",
            "detail": f"{len(window)} transactions within a 10-minute window",
        })
        score += 40

    # Rule 2 – geo mismatch: transaction country != customer home country.
    if txn["country"] != cust["home_country"]:
        signals.append({
            "rule": "geo_mismatch",
            "detail": f"txn country {txn['country']} != home {cust['home_country']}",
        })
        score += 30

    # Rule 3 – amount anomaly: > 3x the trailing average of prior transactions.
    prior = [h["amount"] for h in history
             if datetime.fromisoformat(h["ts"]) < txn_ts]
    if prior:
        avg = sum(prior) / len(prior)
        if avg > 0 and txn["amount"] > 3 * avg:
            signals.append({
                "rule": "amount_anomaly",
                "detail": f"amount {txn['amount']:.2f} > 3x trailing avg {avg:.2f}",
            })
            score += 30

    score = min(score, 100)
    recommendation = (
        "freeze_card_and_open_case" if score >= 70
        else "review" if score >= 40
        else "allow"
    )
    _audit("check_fraud_signals", None, {"txn_id": txn_id}, "ALLOW",
           f"score={score} rec={recommendation}")
    return _ok({
        "txn_id": txn_id,
        "risk_score": score,
        "signals": signals,
        "recommendation": recommendation,
    })


@mcp.tool()
def get_case(case_id: str) -> str:
    """Fetch a dispute/fraud case by ID."""
    con = _conn()
    row = con.execute("SELECT * FROM cases WHERE case_id = ?", (case_id,)).fetchone()
    con.close()
    if not row:
        return _err("not_found", f"No case {case_id}")
    return _ok({"case": dict(row)})


@mcp.tool()
def get_audit_trail(limit: int = 20) -> str:
    """Return the most recent audit-log entries (the compliance evidence trail).
    Use this to show exactly which tools were called, by whom, and whether each
    was allowed or denied."""
    con = _conn()
    rows = con.execute(
        "SELECT ts, tool, actor_role, decision, result FROM audit_log "
        "ORDER BY id DESC LIMIT ?", (limit,)
    ).fetchall()
    con.close()
    return _ok({"entries": [dict(r) for r in rows]})


# ---------------------------------------------------------------------------
# WRITE tools – GUARDED. Require a privileged operator_role or DENIED.
# ---------------------------------------------------------------------------
@mcp.tool()
def freeze_card(card_id: str, reason: str, operator_role: str) -> str:
    """Freeze a card to stop further authorizations. PRIVILEGED: requires an
    operator_role of fraud_ops or supervisor. Idempotent – freezing an
    already-frozen card is a no-op that reports the existing state."""
    if operator_role not in PRIVILEGED_ROLES:
        _audit("freeze_card", operator_role,
               {"card_id": card_id, "reason": reason}, "DENY", "insufficient_role")
        return _err("forbidden",
                    f"Role '{operator_role}' cannot freeze cards. "
                    f"Requires one of: {sorted(PRIVILEGED_ROLES)}")

    con = _conn()
    card = con.execute("SELECT * FROM cards WHERE card_id = ?", (card_id,)).fetchone()
    if not card:
        con.close()
        _audit("freeze_card", operator_role, {"card_id": card_id}, "ALLOW", "not_found")
        return _err("not_found", f"No card {card_id}")

    if card["status"] == "frozen":
        con.close()
        _audit("freeze_card", operator_role,
               {"card_id": card_id, "reason": reason}, "ALLOW", "already_frozen")
        return _ok({"card_id": card_id, "status": "frozen", "idempotent": True,
                     "note": "card was already frozen"})

    con.execute("UPDATE cards SET status = 'frozen' WHERE card_id = ?", (card_id,))
    con.commit()
    con.close()
    _audit("freeze_card", operator_role,
           {"card_id": card_id, "reason": reason}, "ALLOW", "frozen")
    return _ok({"card_id": card_id, "status": "frozen", "idempotent": False,
                 "reason": reason})


@mcp.tool()
def open_dispute_case(customer_id: str, txn_id: str, reason: str,
                       operator_role: str) -> str:
    """Open a dispute/fraud case for a transaction. PRIVILEGED: requires an
    operator_role of fraud_ops or supervisor. Idempotent – if an OPEN case
    already exists for this transaction, it is returned instead of duplicated."""
    if operator_role not in PRIVILEGED_ROLES:
        _audit("open_dispute_case", operator_role,
               {"customer_id": customer_id, "txn_id": txn_id}, "DENY", "insufficient_role")
        return _err("forbidden",
                    f"Role '{operator_role}' cannot open cases. "
                    f"Requires one of: {sorted(PRIVILEGED_ROLES)}")

    con = _conn()
    existing = con.execute(
        "SELECT * FROM cases WHERE txn_id = ? AND status = 'open'", (txn_id,)
    ).fetchone()
    if existing:
        con.close()
        _audit("open_dispute_case", operator_role,
               {"customer_id": customer_id, "txn_id": txn_id}, "ALLOW",
               f"existing {existing['case_id']}")
        return _ok({"case_id": existing["case_id"], "status": "open",
                     "idempotent": True, "note": "an open case already existed"})

    n = con.execute("SELECT COUNT(*) FROM cases").fetchone()[0]
    case_id = f"CASE-{2000 + n + 1}"
    con.execute(
        "INSERT INTO cases (case_id, customer_id, txn_id, reason, status, opened_by, opened_at) "
        "VALUES (?,?,?,?,?,?,?)",
        (case_id, customer_id, txn_id, reason, "open", operator_role,
         datetime.now(timezone.utc).replace(microsecond=0).isoformat()),
    )
    con.commit()
    con.close()
    _audit("open_dispute_case", operator_role,
           {"customer_id": customer_id, "txn_id": txn_id}, "ALLOW", case_id)
    return _ok({"case_id": case_id, "status": "open", "idempotent": False})


# ---------------------------------------------------------------------------
# A read-only RESOURCE (MCP resources = GET-style context, not actions)
# ---------------------------------------------------------------------------
@mcp.resource("novapay://customer/{customer_id}/summary")
def customer_summary(customer_id: str) -> str:
    """Resource view of a customer (masked), for agents that prefer to pull
    context as a resource rather than call a tool."""
    return get_customer(customer_id)


if __name__ == "__main__":
    mcp.run()

Overwriting mcp_server.py


> 🧠 **Exercise — find the lines:** scroll back through the cell above and answer these without running any code:
>
> 1. Which single line decides whether `freeze_card` is allowed to proceed? What does it check?
> 2. Which function masks a card number, and what does it actually return — the whole number, or something else?
> 3. `freeze_card` checks `if card["status"] == "frozen":` *before* doing the update. What would break if that check were removed and an agent retried the same freeze call three times in a row?
>
> Write your answers below.

**Your answers:**

1.
2.
3.


## 4. Validate over the real MCP protocol (no AWS needed)

Now we reseed for a clean baseline and launch a **real** MCP client against the **real** server, asserting on real results — no mocking, anywhere. This is the same idea as a CI regression gate: if any of these checks fail, the server is broken.

Watch what gets checked: tool discovery, that PII never appears raw on the wire, that the fraud burst really does score HIGH while a normal transaction scores LOW, that an unprivileged role is DENIED, that a privileged role succeeds, that retries are idempotent, and that the audit trail actually recorded all of it.

In [8]:
build()  # clean baseline

def _payload(r):
    return json.loads(r.content[0].text)


async def run_checks():
    lines, p, f = [], 0, 0

    def chk(name, ok, detail=""):
        nonlocal p, f
        p += 1 if ok else 0
        f += 0 if ok else 1
        lines.append(("PASS" if ok else "FAIL", name, detail))

    async with stdio_client(_params()) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()

            names = {t.name for t in (await s.list_tools()).tools}
            chk("server advertises all 8 tools",
                {"get_customer", "list_transactions", "get_transaction", "check_fraud_signals",
                 "get_case", "get_audit_trail", "freeze_card", "open_dispute_case"}.issubset(names))

            cust = _payload(await s.call_tool("get_customer", {"customer_id": "CUST-1001"}))
            chk("get_customer returns Amara Okafor", cust["customer"]["name"] == "Amara Okafor")

            raw = json.dumps(cust)
            chk("raw SSN not present (PII masked)", "501-22-8841" not in raw)
            chk("raw PAN not present (PII masked)", "4539112233447788" not in raw)

            hi = _payload(await s.call_tool("check_fraud_signals", {"txn_id": "TXN-1001-F4"}))
            chk("fraud burst scores HIGH (>=70)", hi["risk_score"] >= 70, "score=" + str(hi["risk_score"]))
            chk("velocity+geo+amount all fire",
                {"velocity", "geo_mismatch", "amount_anomaly"}.issubset({x["rule"] for x in hi["signals"]}))

            den = _payload(await s.call_tool("freeze_card",
                {"card_id": "CARD-9001", "reason": "x", "operator_role": "viewer"}))
            chk("freeze_card with viewer is DENIED", den.get("error") == "forbidden")

            fr = _payload(await s.call_tool("freeze_card",
                {"card_id": "CARD-9001", "reason": "fraud", "operator_role": "fraud_ops"}))
            chk("freeze_card with fraud_ops succeeds", fr.get("status") == "frozen")

            ag = _payload(await s.call_tool("freeze_card",
                {"card_id": "CARD-9001", "reason": "fraud", "operator_role": "fraud_ops"}))
            chk("second freeze is idempotent", ag.get("idempotent") is True)

            c1 = _payload(await s.call_tool("open_dispute_case",
                {"customer_id": "CUST-1001", "txn_id": "TXN-1001-F4", "reason": "fraud", "operator_role": "fraud_ops"}))
            chk("open_dispute_case creates a case", bool(c1.get("case_id")))

            c2 = _payload(await s.call_tool("open_dispute_case",
                {"customer_id": "CUST-1001", "txn_id": "TXN-1001-F4", "reason": "fraud", "operator_role": "fraud_ops"}))
            chk("re-open same case is idempotent", c2.get("idempotent") is True)

            tr = _payload(await s.call_tool("get_audit_trail", {"limit": 50}))
            chk("audit trail records ALLOW and DENY",
                {"ALLOW", "DENY"}.issubset({e["decision"] for e in tr["entries"]}))

    for st, name, detail in lines:
        print(st, "-", name, ("(" + detail + ")" if detail else ""))
    return p, f


_p, _f = await run_checks()
result(_p, _f)

PASS - server advertises all 8 tools 
PASS - get_customer returns Amara Okafor 
PASS - raw SSN not present (PII masked) 
PASS - raw PAN not present (PII masked) 
PASS - fraud burst scores HIGH (>=70) (score=100)
PASS - velocity+geo+amount all fire 
PASS - freeze_card with viewer is DENIED 
PASS - freeze_card with fraud_ops succeeds 
PASS - second freeze is idempotent 
PASS - open_dispute_case creates a case 
PASS - re-open same case is idempotent 
PASS - audit trail records ALLOW and DENY 


> 🧠 **Exercise — extend the test:** the checks above never test `list_transactions` or `get_transaction` directly. Sketch (in words, or in a new code cell if you like) one more `chk(...)` line you would add, including what customer/transaction id you would use and what you would assert about the result.

**Your answer / new check sketch:**

_(double-click this cell and write here)_


## 5. Live — Claude (Haiku) drives the same tools on Bedrock

> ⚠️ **Important:** this step needs AWS credentials with access to Amazon Bedrock and the model in `MODEL` enabled in your region. If those aren't configured in this environment, the `try/except` below will catch the failure and print a short message instead of crashing the notebook — everything that makes this lab *production-grade* (the tools, the guard, the masking, the audit trail) was already proven in Step 4 without any AWS dependency at all. Step 5 only swaps in a real model as the decision-maker driving those same tools.

This is the payoff: the model decides **which** tool to call and **with what arguments**; the MCP server still decides whether that call is **allowed** and actually runs it. The model never touches the database directly. Watch the reason → act → observe loop: the agent looks up the customer, reviews transactions, runs fraud checks, and — only if the signals justify it — freezes the card and opens dispute cases, exactly as a human fraud analyst would, through the same guarded tools.

In [9]:
SYSTEM = ("You are NovaPay's fraud-operations assistant. Investigate suspected fraud: look up the customer, "
          "review transactions, run check_fraud_signals before acting. Only freeze a card or open a case when the "
          "signals justify it. When calling a privileged tool pass operator_role=fraud_ops. Be concise.")


def _to_bedrock_tools(mcp_tools):
    return {"tools": [{"toolSpec": {
        "name": t.name,
        "description": (t.description or t.name)[:1000],
        "inputSchema": {"json": t.inputSchema or {"type": "object", "properties": {}}},
    }} for t in mcp_tools]}


async def run_agent(msg, max_turns=8):
    import boto3
    bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
    async with stdio_client(_params()) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            tool_config = _to_bedrock_tools((await s.list_tools()).tools)
            messages = [{"role": "user", "content": [{"text": msg}]}]
            print("User:", msg)
            for _ in range(max_turns):
                resp = bedrock.converse(
                    modelId=MODEL, system=[{"text": SYSTEM}], messages=messages,
                    toolConfig=tool_config, inferenceConfig={"maxTokens": 1024, "temperature": 0},
                )
                out = resp["output"]["message"]
                messages.append(out)
                for b in out["content"]:
                    if "text" in b and b["text"].strip():
                        print("Agent:", b["text"].strip())
                if resp["stopReason"] != "tool_use":
                    return
                results = []
                for b in out["content"]:
                    if "toolUse" not in b:
                        continue
                    tu = b["toolUse"]
                    print("  ->", tu["name"], json.dumps(tu.get("input", {})))
                    res = await s.call_tool(tu["name"], tu.get("input", {}))
                    txt = res.content[0].text if res.content else "{}"
                    print("  <-", txt[:120])
                    results.append({"toolResult": {"toolUseId": tu["toolUseId"], "content": [{"text": txt}]}})
                messages.append({"role": "user", "content": results})


def _has_aws_creds():
    # Cheap check so we don't reseed (and wipe Step 4's audit trail) unless we can
    # actually reach Bedrock. Mirrors the same check module-13-demo.sh uses.
    try:
        import boto3
        boto3.client("sts", region_name=os.environ.get("AWS_REGION", "us-east-1")).get_caller_identity()
        return True
    except Exception:
        return False


if _has_aws_creds():
    build()  # reseed for a clean baseline so the agent freezes the card itself
    try:
        await run_agent("Customer CUST-1001 reported charges they dont recognise. Investigate and act if it is fraud.")
    except Exception as e:
        print("Live run failed even though AWS credentials were found:", str(e)[:200])
else:
    print("No AWS credentials detected -- skipping the live Bedrock run.")
    print("Everything that makes this production-grade (tools, guard, PII masking, audit)")
    print("was already proven in Step 4 without any AWS dependency at all.")

No AWS credentials detected -- skipping the live Bedrock run.
Everything that makes this production-grade (tools, guard, PII masking, audit)
was already proven in Step 4 without any AWS dependency at all.


> 🧠 **Exercise — explain it back:** the `SYSTEM` prompt tells the model to "run check_fraud_signals before acting" and to "only freeze a card or open a case when the signals justify it." Suppose the model ignored that instruction and tried to call `freeze_card` immediately, with `operator_role="fraud_ops"`, before ever calling `check_fraud_signals`. Would the server stop it? Why or why not — and what does your answer tell you about the difference between a *prompt instruction* and a *server-side guard*?

**Your answer:**

_(double-click this cell and write here)_


## 6. The compliance audit trail

Every tool call — allowed or denied — became an immutable row in `audit_log`. This is the PCI/SOX answer to "who did what, when, and was it authorized?" Read it straight back out of SQLite.

In [10]:
con = sqlite3.connect(DB_PATH); con.row_factory = sqlite3.Row
rows = con.execute("SELECT tool, actor_role, decision, result FROM audit_log ORDER BY id").fetchall()
hdr = "tool".ljust(20) + "role".ljust(10) + "decision".ljust(9) + "result"
print(hdr); print("-" * 64)
for r in rows:
    print(r["tool"].ljust(20) + str(r["actor_role"] or "-").ljust(10) + r["decision"].ljust(9) + str(r["result"]))
con.close()

tool                role      decision result
----------------------------------------------------------------
get_customer        -         ALLOW    found
check_fraud_signals -         ALLOW    score=100 rec=freeze_card_and_open_case
freeze_card         viewer    DENY     insufficient_role
freeze_card         fraud_ops ALLOW    frozen
freeze_card         fraud_ops ALLOW    already_frozen
open_dispute_case   fraud_ops ALLOW    CASE-2001
open_dispute_case   fraud_ops ALLOW    existing CASE-2001


> 🧠 **Exercise — explain it back:** why must an audit log like this be **append-only** (rows only ever inserted, never updated or deleted)? What compliance property would break if a `fraud_ops` user were able to edit past `audit_log` rows?

**Your answer:**

_(double-click this cell and write here)_


## 7. Knowledge check

Answer each question yourself first — write down a, b, or c on paper or in the reflection cell below — *before* you run the next cell and reveal the answers and explanations.

**Your answers before revealing:**

Q1: &nbsp;&nbsp;&nbsp; Q2: &nbsp;&nbsp;&nbsp; Q3: &nbsp;&nbsp;&nbsp; Q4:


In [11]:
QUESTIONS = [
    ("Where does the security (authz, PII masking, audit) live in this design?",
     ["In the model prompt", "In the MCP server (the tool layer)", "In the client app"], 1,
     "The model only ASKS; the server DECIDES and acts. That gap is the choke point."),

    ("An agent is prompt-injected to freeze every card and email balances out. Why cannot it succeed?",
     ["The model detects it",
      "No such tool exists, the role check runs, only masked data exists – and it is audited",
      "The DB blocks it"], 1,
     "Least privilege + governed tools mean a compromised model cannot exceed a fraud_ops human."),

    ("Why are freeze_card and open_dispute_case idempotent?",
     ["To save storage", "So an agent that retries cannot double-act", "For speed"], 1,
     "Agents retry; idempotency makes retries safe (freezing a frozen card is a no-op)."),

    ("Why does the MCP server run as its own process launched by the client?",
     ["To use more CPU", "Because MCP stdio transport is process-to-process by design", "Random"], 1,
     "With stdio the server is its own process; over HTTP it would be a networked service."),
]

for i, (q, opts, ans, ex) in enumerate(QUESTIONS, 1):
    print("Q" + str(i) + ".", q)
    for j, o in enumerate(opts):
        print("   " + chr(97 + j) + ")", o)
    print("   Answer:", chr(97 + ans) + ")", opts[ans])
    print("   ->", ex)
    print()

Q1. Where does the security (authz, PII masking, audit) live in this design?
   a) In the model prompt
   b) In the MCP server (the tool layer)
   c) In the client app
   Answer: b) In the MCP server (the tool layer)
   -> The model only ASKS; the server DECIDES and acts. That gap is the choke point.

Q2. An agent is prompt-injected to freeze every card and email balances out. Why cannot it succeed?
   a) The model detects it
   b) No such tool exists, the role check runs, only masked data exists – and it is audited
   c) The DB blocks it
   Answer: b) No such tool exists, the role check runs, only masked data exists – and it is audited
   -> Least privilege + governed tools mean a compromised model cannot exceed a fraud_ops human.

Q3. Why are freeze_card and open_dispute_case idempotent?
   a) To save storage
   b) So an agent that retries cannot double-act
   c) For speed
   Answer: b) So an agent that retries cannot double-act
   -> Agents retry; idempotency makes retries safe (fre

## Wrap-up — revisit your first answer

Scroll back to the very first reflection cell, where you wrote down what "the model never holds credentials" means. Now that you've built the server, guarded its write tools, masked its PII, watched a live agent get stopped by (and succeed within) those same guards, and read the audit trail those actions left behind — has your answer changed? Rewrite it below in one paragraph, as if you were explaining it to a colleague who has never heard of MCP.

**Your final explanation:**

_(double-click this cell and write here)_
